# Neuro-Symbolic Text-to-FOL Reasoning Datasets Demo

## Overview

This notebook demonstrates the **ProofWriter + FOLIO datasets** standardized for neuro-symbolic text-to-FOL reasoning evaluation.

### Datasets Included

1. **ProofWriter** (Tafjord et al. ACL-IJCNLP 2021): 60,000 examples of natural language logical entailment with gold proof traces. Features reasoning depth (0–5), fact/rule counts, and structured proof annotations.

2. **FOLIO** (Han et al. EMNLP 2024): 1,204 examples of NL reasoning with gold FOL annotations (∀, →, ¬, ∨). Features formal FOL premises and conclusions verified by a FOL inference engine.

Both datasets are standardized to the **exp_sel_data_out schema** with unified field structure:
- `input`: formatted context + query string
- `output`: 'true' or 'false' label
- `metadata_*`: structured annotations (depth, fact/rule counts, FOL forms, proof traces, etc.)

This demo loads a curated subset (6 examples, 3 per dataset) to explore the schema and understand the data structure.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-Colab packages (always install)
# None needed for this demo - all imports are pre-installed

# Core packages (install locally only, skip on Colab)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

print("Dependencies installed successfully.")

In [ ]:
import json
import os
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")

In [ ]:
# Data loading helper function
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-1f4229-typed-unification-failure-recovery-towar/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load mini_demo_data.json from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local file")

In [ ]:
# Load the data
data = load_data()
print(f"✓ Loaded data successfully")
print(f"  Metadata: {data['metadata']}")

## Configuration

Define tunable parameters for data exploration. These are set to minimal values for quick demo runs. Adjust as needed.

In [ ]:
# Configuration: Minimal values for demo
MAX_EXAMPLES_PER_DATASET = 3
SHOW_INPUT_LENGTH = 200
SHOW_METADATA_FIELDS = [
    'metadata_dataset_id',
    'metadata_task_type',
    'metadata_split',
    'metadata_reasoning_depth',
    'metadata_num_facts',
    'metadata_num_rules',
    'output'
]

print(f"Config: MAX_EXAMPLES={MAX_EXAMPLES_PER_DATASET}, SHOW_INPUT_LENGTH={SHOW_INPUT_LENGTH}")

## Data Structure Overview

The dataset has two levels:
1. **Top-level metadata**: description, datasets_included, total_examples
2. **Datasets array**: list of {dataset, examples}
3. **Each example**: input, output, metadata_* fields

In [ ]:
# Extract structure info
datasets_included = data['metadata']['datasets_included']
total_examples_in_full = data['metadata']['total_examples']

print(f"\nFull dataset stats (from metadata):")
print(f"  Total examples: {total_examples_in_full}")
print(f"  Datasets included: {', '.join(datasets_included)}")

# Count examples in this demo
demo_examples = []
for ds in data['datasets']:
    dataset_name = ds['dataset']
    examples = ds['examples']
    demo_examples.append((dataset_name, examples))
    print(f"\nDemo subset - {dataset_name}: {len(examples)} examples")

## Explore ProofWriter Examples

ProofWriter features logical entailment with gold proof traces. Each example has:
- **Theory**: Natural language facts and rules
- **Query**: Question to verify
- **Answer**: True/False/Unknown
- **Gold proof**: Structured proof trace showing which facts/rules led to the answer

In [ ]:
# Extract ProofWriter examples
proofwriter_examples = [ds[1] for ds in demo_examples if ds[0] == 'proofwriter'][0]

print(f"\n{'='*80}")
print(f"PROOFWRITER EXAMPLE 1")
print(f"{'='*80}")

ex = proofwriter_examples[0]
print(f"\nQuestion (from metadata):")
print(f"  {ex['metadata_question']}")
print(f"\nAnswer: {ex['output']}")
print(f"\nReasoning depth: {ex['metadata_reasoning_depth']}")
print(f"Facts: {ex['metadata_num_facts']}, Rules: {ex['metadata_num_rules']}")
print(f"\nTheory (first 500 chars):")
print(f"  {ex['metadata_source_text'][:500]}...")
print(f"\nGold proof (first 300 chars):")
print(f"  {ex['metadata_gold_proof'][:300]}...")

In [ ]:
# Show all ProofWriter examples as table
print(f"\n{'='*80}")
print(f"ALL PROOFWRITER EXAMPLES (Demo)")
print(f"{'='*80}\n")

pw_table = []
for i, ex in enumerate(proofwriter_examples[:MAX_EXAMPLES_PER_DATASET], 1):
    pw_table.append({
        'ID': i,
        'Question': ex['metadata_question'][:60] + '...' if len(ex['metadata_question']) > 60 else ex['metadata_question'],
        'Answer': ex['output'],
        'Depth': ex['metadata_reasoning_depth'],
        'Facts': ex['metadata_num_facts'],
        'Rules': ex['metadata_num_rules'],
    })

pw_df = pd.DataFrame(pw_table)
print(pw_df.to_string(index=False))

## Explore FOLIO Examples

FOLIO features FOL-grounded NL reasoning. Each example has:
- **Premises**: Natural language premises
- **Premises-FOL**: Formal first-order logic translation (∀, →, ¬, ∨)
- **Conclusion**: NL conclusion to verify
- **Conclusion-FOL**: Formal FOL form of conclusion
- **Label**: True/False verified by FOL inference engine

In [ ]:
# Extract FOLIO examples
folio_examples = [ds[1] for ds in demo_examples if ds[0] == 'folio'][0]

print(f"\n{'='*80}")
print(f"FOLIO EXAMPLE 1")
print(f"{'='*80}")

ex = folio_examples[0]
print(f"\nConclusion (NL):")
print(f"  {ex['metadata_question']}")
print(f"\nAnswer: {ex['output']}")
print(f"\nPremises (NL, first 200 chars):")
print(f"  {ex['metadata_source_text'][:200]}...")
print(f"\nPremises-FOL (first 200 chars):")
print(f"  {ex['metadata_premises_fol'][:200]}...")
print(f"\nConclusion-FOL:")
print(f"  {ex['metadata_conclusion_fol']}")

In [ ]:
# Show all FOLIO examples as table
print(f"\n{'='*80}")
print(f"ALL FOLIO EXAMPLES (Demo)")
print(f"{'='*80}\n")

folio_table = []
for i, ex in enumerate(folio_examples[:MAX_EXAMPLES_PER_DATASET], 1):
    folio_table.append({
        'ID': i,
        'Conclusion': ex['metadata_question'][:60] + '...' if len(ex['metadata_question']) > 60 else ex['metadata_question'],
        'Answer': ex['output'],
        'Depth': ex['metadata_reasoning_depth'],
        'Facts': ex['metadata_num_facts'],
        'Rules': ex['metadata_num_rules'],
    })

folio_df = pd.DataFrame(folio_table)
print(folio_df.to_string(index=False))

## Schema: Extracted Facts

Both datasets include `metadata_extracted_facts_json`, which parses the NL premises/theory into structured facts and rules.

In [ ]:
# Show fact extraction for a ProofWriter example
ex = proofwriter_examples[0]
facts = json.loads(ex['metadata_extracted_facts_json'])

print(f"\nExtracted facts from ProofWriter Example 1:")
print(f"Total: {len(facts)} (facts + rules)")

fact_count = sum(1 for f in facts if not f['is_rule'])
rule_count = sum(1 for f in facts if f['is_rule'])

print(f"  Facts: {fact_count}")
print(f"  Rules: {rule_count}")

print(f"\nFirst 3 facts/rules:")
for i, f in enumerate(facts[:3], 1):
    fact_type = "RULE" if f['is_rule'] else "FACT"
    print(f"  {i}. [{fact_type}] {f['predicate'][:70]}...")

## Visualization: Reasoning Complexity

Compare reasoning depth and fact/rule counts across the demo examples.

In [ ]:
# Collect stats for visualization
all_examples = []
for dataset_name, examples in demo_examples:
    for ex in examples:
        all_examples.append({
            'dataset': dataset_name,
            'depth': ex['metadata_reasoning_depth'],
            'facts': ex['metadata_num_facts'],
            'rules': ex['metadata_num_rules'],
            'answer': ex['output'],
            'task_type': ex['metadata_task_type'],
        })

stats_df = pd.DataFrame(all_examples)

print("\nDataset Statistics (Demo Subset):")
print(f"\n{stats_df.groupby('dataset')[['depth', 'facts', 'rules']].agg(['mean', 'min', 'max']).round(2)}")

In [ ]:
# Plot reasoning complexity
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Reasoning Complexity in Demo Subset', fontsize=14, fontweight='bold')

# Plot 1: Depth distribution
for dataset in stats_df['dataset'].unique():
    subset = stats_df[stats_df['dataset'] == dataset]
    axes[0].scatter(subset.index, subset['depth'], label=dataset, s=100, alpha=0.7)
axes[0].set_xlabel('Example Index')
axes[0].set_ylabel('Reasoning Depth')
axes[0].set_title('Reasoning Depth')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot 2: Fact count
for dataset in stats_df['dataset'].unique():
    subset = stats_df[stats_df['dataset'] == dataset]
    axes[1].scatter(subset.index, subset['facts'], label=dataset, s=100, alpha=0.7, marker='s')
axes[1].set_xlabel('Example Index')
axes[1].set_ylabel('Number of Facts')
axes[1].set_title('Fact Count')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Plot 3: Rule count
for dataset in stats_df['dataset'].unique():
    subset = stats_df[stats_df['dataset'] == dataset]
    axes[2].scatter(subset.index, subset['rules'], label=dataset, s=100, alpha=0.7, marker='^')
axes[2].set_xlabel('Example Index')
axes[2].set_ylabel('Number of Rules')
axes[2].set_title('Rule Count')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Visualization complete.")

## Summary

This demo showcases the **ProofWriter + FOLIO neuro-symbolic reasoning datasets**:

### Key Takeaways

1. **Unified Schema**: Both datasets fit the `exp_sel_data_out` schema with common fields (input, output, metadata_*)

2. **ProofWriter**: ~60k examples of logical entailment with gold proof traces, reasoning depth 0–5

3. **FOLIO**: ~1.2k examples of FOL-grounded reasoning with formal premise/conclusion annotations

4. **Rich Metadata**: Each example includes:
   - Extracted facts (atoms + rules from NL)
   - Reasoning depth
   - Gold proof traces (ProofWriter) / FOL annotations (FOLIO)
   - Source text and normalized labels

5. **Use Cases**:
   - Train neuro-symbolic reasoners (text → FOL → logical inference)
   - Evaluate fact extraction precision/recall
   - Measure multi-hop reasoning accuracy
   - Compare against neural baselines (RAG, chain-of-thought)

Full dataset (61k examples) is split into 5 JSON files (≤60MB each) for practical loading.

In [ ]:
print("\n" + "="*80)
print("DEMO COMPLETE")
print("="*80)
print(f"\nLoaded {len(stats_df)} examples from ProofWriter + FOLIO datasets.")
print(f"Full dataset contains {total_examples_in_full} examples across {len(datasets_included)} datasets.")
print(f"\nTo use the full dataset:")
print(f"  1. Load from full_data_out_*.json files (chunked for size)")
print(f"  2. Parse JSON and iterate over examples")
print(f"  3. Access metadata_* fields for structured reasoning info")
print(f"\nSee source code (data.py) for dataset conversion logic.")